# Tanglish offensive-language detection: Kaggle runner

**Settings:** Accelerator = **GPU T4 x2**, Internet = **on**.

Run one stage per session with *Save Version → Save & Run All (Commit)*, which keeps running for up to 12 h
without the browser open. Everything under `/kaggle/working` is saved with the version.

To continue in a new session, attach the previous version's output
(*Add Input → Your Work → this notebook*) and run the restore cell. Finished runs are skipped
and interrupted ones resume from their last epoch.

| stage | what | ≈ GPU-h |
|---|---|---|
| 1 | backbone sweep (mBERT, XLM-R, MuRIL, IndicBERTv2) | 2.5 |
| 2 | domain-adaptive pretraining (MuRIL, XLM-R) + fine-tune | 4 |
| 3 | main ablation on the chosen backbone, 3 seeds | 11 |
| 4 | large backbones, 2 seeds | 8 |

In [ ]:
import os
REPO = "https://github.com/JaiivantArvind/Tanglish-Hate-Speech-Detection.git"
BRANCH = "rebuild-hierarchical-pipeline"   # set to "main" once the branch is merged
WORK = "/kaggle/working/tanglish-hsd"
if not os.path.isdir(WORK):
    !git clone -q --branch {BRANCH} {REPO} {WORK}
else:
    !git -C {WORK} fetch -q origin {BRANCH} && git -C {WORK} checkout -q {BRANCH} && git -C {WORK} pull -q
%cd {WORK}
!pip install -q -r requirements.txt
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
# Restore finished runs, DAPT models and caches from an attached previous version.
# Kaggle mounts notebook output at /kaggle/input/notebooks/<user>/<slug>/, so search recursively.
import glob, shutil
srcs = glob.glob("/kaggle/input/**/tanglish-hsd", recursive=True)
print("attached outputs:", srcs or "none - use Add Input -> Your Work")
for src in srcs:
    for sub in ("runs", "dapt", "data"):
        if os.path.isdir(f"{src}/{sub}"):
            shutil.copytree(f"{src}/{sub}", sub, dirs_exist_ok=True)
            print("restored", f"{src}/{sub}")
print("runs present:", sorted(os.listdir("runs")) if os.path.isdir("runs") else "none")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/phase1_data.py
!python scripts/phase2_features.py
!python -m pytest tests -q

## Run a stage
Before stage 3, look at the stage 1–2 table. If DAPT did not beat plain MuRIL on **dev** macro-F1,
point `configs/s3_base.yaml` at the best stage-1 backbone (or pass `--set model_name=...`).
Before stage 4, set `configs/s4_base.yaml` to the stage-3 winner's head, lexicon and translit settings.

In [ ]:
STAGE = 1
ONLY = ""   # optional: space-separated names, e.g. "dapt_xlmr s2_xlmr_base_dapt"
only_flag = f"--only {ONLY}" if ONLY else ""
!python scripts/run_experiments.py --stage {STAGE} {only_flag}


In [ ]:
!python -m tanglish.evaluate summarize --runs runs --out reports/results_table.md

## Final report (after stage 3 or 4)

In [ ]:
!python scripts/phase5_report.py --baseline s3_cls --proposed s3_hier_tree
from IPython.display import Markdown, Image, display
display(Markdown(open("reports/results.md").read()))
for f in sorted(glob.glob("reports/*.png")):
    display(Image(f, width=700))